## BMERA research experiments

Enable the RUN flags in the relevant sections to run the longer sweeps.

### 1. Setup

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
import pennylane as qml
from pennylane import numpy as qnp
from matplotlib import pyplot as plt

from bmera_core import (
    build_bmera_meta,
    compute_expval,
    compute_expval_real,
    compute_all_expval,
    model_output,
    forward_single,
    forward_batch,
    forward_single_with_cache,
    forward_batch_with_cache,
    loss,
    loss_and_grad_autodiff_microbatched,
    loss_and_grad_cached,
    train_step_cached,
    train_step,
    benchmark_training_scaling,
    train_loop,
    layerwise_param_mask,
    compute_grad,
    compute_model_grad,
    weights,
)


# Default for headless/automated runs
SAVE_FIGURES = False

from bmera.experiments import snapshots as qsb
from bmera.benchmarking import benchmark_callable, benchmark_context


In [ ]:
print('JAX devices:', jax.devices())

### 2. Forward Pass Demo

In [ ]:
a = 2
n_qubits = 2**a
n_batch = 4
params = np.random.uniform(-np.pi, np.pi, (a * n_qubits, 4))
in_data = np.random.normal(0, 1, (n_qubits))
in_data_batch = np.random.normal(0, 1, (n_batch, n_qubits))
expvals = compute_all_expval(1, params, in_data, n_qubits)
#expvals[:4], expvals.shape

In [ ]:
in_data_batch = in_data_batch.reshape((-1, n_qubits))
in_data_batch

In [ ]:
meta = build_bmera_meta(n_qubits=n_qubits, type=1)
forward_batch(params, in_data_batch, meta)

### 3. Gradient Demo + Finite-Difference Sanity Check

In [ ]:
def finite_diff_grad(fn, x, eps=2e-6):
    x = np.array(x, dtype=float)
    g = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        xp = x.copy()
        xm = x.copy()
        xp[idx] += eps
        xm[idx] -= eps
        g[idx] = (fn(xp) - fn(xm)) / (2 * eps)
        it.iternext()
    return g

def rel_err(a, b):
    return np.linalg.norm((a - b).ravel()) / (np.linalg.norm(a.ravel()) + np.linalg.norm(b.ravel()) + 1e-12)

type_ = 1
qubit = 1
g_ad = np.array(compute_grad(type_, params, in_data, n_qubits, qubit))
g_fd = finite_diff_grad(lambda p: float(compute_expval_real(type_, p, in_data, n_qubits, qubit)), params)
rel_err(g_ad, g_fd)

In [ ]:
w = np.random.uniform(0, 1, n_qubits)
g_params, g_weights = compute_model_grad(type_, params, w, in_data, n_qubits)
np.linalg.norm(np.array(g_params)), np.linalg.norm(np.array(g_weights))

### 4. Concise Pennylane Parity Check

In [ ]:
from bmera.backends.pennylane import pennylane_bmera_expvals


a = 2
n_qubits = 2**a
params_small = np.random.uniform(-np.pi, np.pi, (a * n_qubits, 4))
in_data_small = np.random.normal(0, 1, n_qubits)
tn_out = np.array(compute_all_expval(1, params_small, in_data_small, n_qubits))
pl_out = pennylane_bmera_expvals(in_data_small, n_qubits, 1, params_small)
float(np.max(np.abs(tn_out - pl_out)))

### 5. Time Measurement (Consolidated)

In [ ]:
RUN_TIMING_BENCHMARKS = False

In [ ]:
if RUN_TIMING_BENCHMARKS:
    n_list = np.array([4, 8, 16, 32])
    timing_rows = []
    rng = np.random.default_rng(10)
    forward_time = []
    forward_time_std = []
    grad_expval_time = []
    grad_expval_time_std = []
    grad_model_time = []
    grad_model_time_std = []

    for n_qubits in n_list:
        a = int(np.log2(n_qubits))
        params = rng.uniform(-np.pi, np.pi, (a * n_qubits, 4))
        in_data = rng.normal(0, 1, n_qubits)
        w = rng.uniform(0, 1, n_qubits)
        timing_context = benchmark_context(jax.devices()[0], jnp.asarray(params).dtype, 3, 1, 10)

        timing = benchmark_callable(lambda: compute_all_expval(1, params, in_data, n_qubits), repeats=3, warmup=1)
        timing_rows.append(dict(n_qubits=int(n_qubits), operation="forward", **timing_context, **timing))
        m, s = timing["steady_mean_ms"], timing["steady_std_ms"]
        forward_time.append(m); forward_time_std.append(s)

        timing = benchmark_callable(lambda: compute_grad(1, params, in_data, n_qubits, 1), repeats=3, warmup=1)
        timing_rows.append(dict(n_qubits=int(n_qubits), operation="expval_gradient", **timing_context, **timing))
        m, s = timing["steady_mean_ms"], timing["steady_std_ms"]
        grad_expval_time.append(m); grad_expval_time_std.append(s)

        timing = benchmark_callable(lambda: compute_model_grad(1, params, w, in_data, n_qubits), repeats=3, warmup=1)
        timing_rows.append(dict(n_qubits=int(n_qubits), operation="model_gradient", **timing_context, **timing))
        m, s = timing["steady_mean_ms"], timing["steady_std_ms"]
        grad_model_time.append(m); grad_model_time_std.append(s)

    forward_time = np.array(forward_time)
    forward_time_std = np.array(forward_time_std)
    grad_expval_time = np.array(grad_expval_time)
    grad_expval_time_std = np.array(grad_expval_time_std)
    grad_model_time = np.array(grad_model_time)
    grad_model_time_std = np.array(grad_model_time_std)

    forward_time, grad_expval_time, grad_model_time

In [ ]:
if RUN_TIMING_BENCHMARKS:
    plt.figure(figsize=(6, 4))
    plt.errorbar(n_list, forward_time, forward_time_std, label='BMERA simulation')
    plt.xscale('log'); plt.yscale('log')
    plt.xlabel('N'); plt.ylabel('time per run, ms')
    plt.legend()
    if SAVE_FIGURES:
        plt.savefig('forward_time')
        plt.savefig('forward_time_upd')
    plt.show()

In [ ]:
if RUN_TIMING_BENCHMARKS:
    plt.figure(figsize=(6, 4))
    plt.errorbar(n_list, grad_model_time, grad_model_time_std, label='BMERA simulation')
    plt.xscale('log'); plt.yscale('log')
    plt.xlabel('N'); plt.ylabel('time per run backwards, model, ms')
    plt.legend()
    if SAVE_FIGURES:
        plt.savefig('model_backwards_time')
        plt.savefig('model_backwards_time_upd')
    plt.show()

### 6. Research Benchmarks: Autodiff vs Cached Gradient
This section compares scaling and training behavior for the baseline autodiff path and the cached BMERA-specific gradient path.


In [ ]:
if RUN_TIMING_BENCHMARKS:
    N_SWEEP = [4, 8, 16, 32]
    BATCH_SWEEP = (1, 16)

    rows_auto_cpu = benchmark_training_scaling(
        N_SWEEP, batch_sizes=BATCH_SWEEP, type=1, repeats=3, warmup=1, grad_engine='autodiff', device='cpu', seed=10
    )
    rows_cached_cpu = benchmark_training_scaling(
        N_SWEEP, batch_sizes=BATCH_SWEEP, type=1, repeats=3, warmup=1, grad_engine='cached', device='cpu', seed=10
    )

In [ ]:
if RUN_TIMING_BENCHMARKS:
    def get_jax_gpu_devices():
        return [device for device in jax.devices() if device.platform in ('gpu', 'cuda', 'rocm')]

    _gpu_devices = get_jax_gpu_devices()

    if _gpu_devices:
        rows_auto_gpu = benchmark_training_scaling(
            N_SWEEP, batch_sizes=BATCH_SWEEP, type=1, repeats=3, warmup=1, grad_engine='autodiff', device='gpu', seed=10
        )
        rows_cached_gpu = benchmark_training_scaling(
            N_SWEEP, batch_sizes=BATCH_SWEEP, type=1, repeats=3, warmup=1, grad_engine='cached', device='gpu', seed=10
        )
    else:
        _platforms = sorted({device.platform for device in jax.devices()})
        print(
            'Skipping GPU benchmark rows because no JAX GPU backend is available. '
            f'Visible JAX platforms: {_platforms}.'
        )
        rows_auto_gpu = []
        rows_cached_gpu = []

    bench_rows = rows_auto_cpu + rows_cached_cpu + rows_auto_gpu + rows_cached_gpu
    bench_rows[:2]


In [ ]:
if RUN_TIMING_BENCHMARKS:
    def pick_series(rows, engine, batch_size, device, key):
        subset = [r for r in rows if r['grad_engine'] == engine and r['batch_size'] == batch_size and r['device'] == device]
        subset = sorted(subset, key=lambda r: r['n_qubits'])
        x = np.array([r['n_qubits'] for r in subset])
        y = np.array([r[key] for r in subset])
        return x, y

    rows_all = rows_auto_cpu + rows_cached_cpu
    x1, y1 = pick_series(rows_all, 'autodiff', 16, 'cpu', 'grad_steady_ms')
    x2, y2 = pick_series(rows_all, 'cached', 16, 'cpu', 'grad_steady_ms')

    plt.figure(figsize=(6, 4))
    plt.plot(x1, y1, marker='o', label='autodiff grad (CPU, batch=16)')
    plt.plot(x2, y2, marker='o', label='cached grad (CPU, batch=16)')
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('n_qubits')
    plt.ylabel('gradient time, ms')
    plt.legend()
    plt.show()


In [ ]:
if RUN_TIMING_BENCHMARKS:
    rows_all_gpu = rows_auto_gpu + rows_cached_gpu
    x1, y1 = pick_series(rows_all_gpu, 'autodiff', 16, 'gpu', 'grad_steady_ms')
    x2, y2 = pick_series(rows_all_gpu, 'cached', 16, 'gpu', 'grad_steady_ms')

    plt.figure(figsize=(6, 4))
    plt.plot(x1, y1, marker='o', label='autodiff grad (GPU, batch=16)')
    plt.plot(x2, y2, marker='o', label='cached grad (GPU, batch=16)')
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('n_qubits')
    plt.ylabel('gradient time, ms')
    plt.legend()
    plt.show()

In [ ]:
if RUN_TIMING_BENCHMARKS:
    n_qubits = 16
    type_ = 1
    meta = build_bmera_meta(n_qubits, type_)
    a = int(np.log2(n_qubits))
    params0 = jnp.array(np.random.uniform(-0.4, 0.4, (a * n_qubits, 4)).astype(np.float32))
    head0 = jnp.array(np.random.uniform(-0.4, 0.4, (n_qubits,)).astype(np.float32))
    X = jnp.array(np.random.normal(0, 1, (32, n_qubits)).astype(np.float32))
    y = jnp.array(np.random.uniform(-1, 1, (32,)).astype(np.float32))

    res_auto = train_loop(params0, head0, X, y, meta, steps=20, lr=1e-2, grad_engine='autodiff')
    res_cached = train_loop(params0, head0, X, y, meta, steps=20, lr=1e-2, grad_engine='cached')
    mask = layerwise_param_mask(meta, max_unfrozen_layer=1)
    res_cached_warm = train_loop(params0, head0, X, y, meta, steps=20, lr=1e-2, grad_engine='cached', param_mask=mask)


In [ ]:
if RUN_TIMING_BENCHMARKS:

    plt.figure(figsize=(6, 4))
    plt.plot(res_auto['losses'], label='autodiff')
    plt.plot(res_cached['losses'], label='cached')
    plt.plot(res_cached_warm['losses'], label='cached + layerwise warm start')
    plt.xlabel('step')
    plt.ylabel('loss')
    plt.legend()
    plt.show()


### 7. Synthetic Fixed-Information Dataset Benchmarks

This section benchmarks BMERA on synthetic binary tasks (`moons`, `circles`, `xor`, and `parity`). The comparison is a fixed-information control: each qubit count receives the same informative features, then the remaining input dimensions are zero-padded. Training uses full gradient batches chosen from the updated GPU throughput sweep, not microbatch accumulation.


In [ ]:
from bmera.experiments.synthetic import (
    SYNTHETIC_BATCH_SIZE_BY_QUBITS,
    SYNTHETIC_DATASETS,
    SYNTHETIC_LR_BY_OPTIMIZER,
    SYNTHETIC_OPTIMIZERS,
    SYNTHETIC_QUBITS,
    assert_synthetic_dataset_ok,
    prepare_synthetic_dataset,
    print_synthetic_results_table,
    run_synthetic_bmera_sweep,
    select_best_synthetic_results,
    train_synthetic_bmera_once,
)

In [ ]:
# Smoke test: validates preprocessing and runs two no-microbatch training steps for each optimizer.
smoke_prepared = prepare_synthetic_dataset('xor', 8)
assert_synthetic_dataset_ok(smoke_prepared)
for smoke_optimizer in SYNTHETIC_OPTIMIZERS:
    smoke_result = train_synthetic_bmera_once(
        'xor',
        8,
        repeat=0,
        steps=2,
        lr=SYNTHETIC_LR_BY_OPTIMIZER[smoke_optimizer],
        optimizer=smoke_optimizer,
        log_every=1,
        save_weights=False,
    )
    assert smoke_result['optimizer'] == smoke_optimizer
    assert smoke_result['microbatch_size'] is None
    assert smoke_result['train_batch_size'] == SYNTHETIC_BATCH_SIZE_BY_QUBITS[8]
    print(
        'Synthetic smoke test passed: '
        f"optimizer={smoke_optimizer}, final_loss={smoke_result['final_loss']:.4f}, "
        f"val_acc={smoke_result['val_accuracy']:.3f}, "
        f"test_acc={smoke_result['test_accuracy']:.3f}"
    )



In [ ]:

# Full synthetic sweep.
# This launches 4 datasets x 2 qubit counts x 2 optimizers x 3 repeats.
# It takes about 72 mins to run
# Set RUN_SYNTHETIC_FULL_SWEEP=True after the smoke test passes.
RUN_SYNTHETIC_FULL_SWEEP = False

if RUN_SYNTHETIC_FULL_SWEEP:
    synthetic_results = run_synthetic_bmera_sweep()
    synthetic_best_results = select_best_synthetic_results(synthetic_results)
    print_synthetic_results_table(synthetic_results)
else:
    synthetic_results = []
    synthetic_best_results = {}
    print('Synthetic full sweep skipped. Set RUN_SYNTHETIC_FULL_SWEEP=True to run all datasets/qubit counts.')



In [ ]:
# Synthetic benchmark plots. Run the full sweep cell first to populate synthetic_results.
if not synthetic_results:
    print('No synthetic sweep results to plot yet.')
else:
    synthetic_best_results = select_best_synthetic_results(synthetic_results)
    print_synthetic_results_table(synthetic_results)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for dataset_name in SYNTHETIC_DATASETS:
        for optimizer in SYNTHETIC_OPTIMIZERS:
            qubits = []
            val_acc = []
            test_acc = []
            for n_qubits in SYNTHETIC_QUBITS:
                result = synthetic_best_results.get((dataset_name, int(n_qubits), optimizer))
                if result is None:
                    continue
                qubits.append(int(n_qubits))
                val_acc.append(result['val_accuracy'])
                test_acc.append(result['test_accuracy'])
            if qubits:
                label = f'{dataset_name} ({optimizer})'
                linestyle = '-' if optimizer == 'adam' else '--'
                axes[0].plot(qubits, val_acc, marker='o', linestyle=linestyle, label=label)
                axes[1].plot(qubits, test_acc, marker='o', linestyle=linestyle, label=label)

    for ax, title in zip(axes, ('Validation accuracy', 'Test accuracy')):
        ax.set_xscale('log', base=2)
        ax.set_xticks(SYNTHETIC_QUBITS)
        ax.set_xticklabels([str(q) for q in SYNTHETIC_QUBITS])
        ax.set_ylim(0.0, 1.02)
        ax.set_xlabel('qubits / input size')
        ax.set_ylabel('accuracy')
        ax.set_title(title)
        ax.grid(True, alpha=0.25)
        ax.legend()
    plt.tight_layout()
    plt.show()

    for dataset_name in SYNTHETIC_DATASETS:
        plt.figure(figsize=(7, 4))
        plotted = False
        for n_qubits in SYNTHETIC_QUBITS:
            for optimizer in SYNTHETIC_OPTIMIZERS:
                result = synthetic_best_results.get((dataset_name, int(n_qubits), optimizer))
                if result is None or result['losses'].size == 0:
                    continue
                linestyle = '-' if optimizer == 'adam' else '--'
                plt.plot(result['losses'], linestyle=linestyle, label=f'{n_qubits} qubits ({optimizer})')
                plotted = True
        if plotted:
            plt.title(f'{dataset_name}: best-repeat training loss')
            plt.xlabel('step')
            plt.ylabel('MSE loss')
            plt.xscale("log")
            plt.legend()
            plt.tight_layout()
            plt.savefig(f'{dataset_name}_best_repeat_training_loss_logscale.png')
        else:
            plt.close()



#### 7.1 Leakage Control: Analytic TFIM Descriptors

This legacy control intentionally retains the original Bogoliubov-angle task:

\[
\theta_k=\frac{1}{2}\operatorname{atan2}\left(\sin k,\;g-\cos k\right).
\]

The descriptors are deterministic functions of the same scalar \(g\) that defines
the label through \(g<1\) or \(g>1\). They are therefore useful as a leakage
control, but not as evidence that BMERA learned entanglement or a uniquely quantum
phase representation. The primary physical benchmark is now Section 7.2, where
models receive finite-shot local measurement records from held-out ground states.


In [ ]:
from bmera.experiments.tfim import (
    assert_tfim_dataset_ok,
    make_tfim_phase_dataset,
    run_tfim_bmera_sweep,
    tfim_bogoliubov_angles,
)

In [ ]:
# Dataset sanity check and physical feature preview.
tfim_preview = make_tfim_phase_dataset(8, n_train=512, n_val=128, n_test=128, n_critical_test=128)
assert_tfim_dataset_ok(tfim_preview)

representative_g = np.array([0.4, 0.8, 0.95, 1.05, 1.2, 1.6], dtype=np.float32)
representative_angles, representative_k = tfim_bogoliubov_angles(representative_g, 64)
plt.figure(figsize=(7, 4))
for g_value, angles in zip(representative_g, representative_angles):
    plt.plot(representative_k / np.pi, angles, label=f'g={g_value:.2f}')
plt.xlabel(r'$k/\pi$')
plt.ylabel(r'Bogoliubov angle $\theta_k$')
plt.title('TFIM ground-state descriptors across the quantum critical point')
plt.grid(True, alpha=0.25)
plt.legend(ncol=2)
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig('tfim_bogoliubov_angle_features.png', dpi=160)
plt.show()


In [ ]:
# Linear leakage control for the analytic TFIM descriptors.
# Large legacy BMERA sweeps are disabled; this section demonstrates why the
# descriptor task is not used as the primary physical result.
tfim_leakage_control_rows = []
for leakage_n_qubits in (8, 16, 32, 64):
    leakage_dataset = make_tfim_phase_dataset(leakage_n_qubits)
    leakage_result = qsb.analytic_tfim_ridge_leakage_control(
        leakage_dataset['train']['X'],
        leakage_dataset['train']['y'],
        {
            'validation': (
                leakage_dataset['val']['X'],
                leakage_dataset['val']['y'],
            ),
            'iid_test': (
                leakage_dataset['test']['X'],
                leakage_dataset['test']['y'],
            ),
            'critical_test': (
                leakage_dataset['critical_test']['X'],
                leakage_dataset['critical_test']['y'],
            ),
        },
    )
    tfim_leakage_control_rows.append({
        'n_qubits': leakage_n_qubits,
        **leakage_result,
    })

print('Analytic TFIM ridge leakage control')
print('qubits | validation | iid test | near-critical test')
for row in tfim_leakage_control_rows:
    print(
        f"{row['n_qubits']:>6d} | {row['validation']:.3f} | "
        f"{row['iid_test']:.3f} | {row['critical_test']:.3f}"
    )

RUN_LEGACY_TFIM_BMERA_SWEEP = False
if RUN_LEGACY_TFIM_BMERA_SWEEP:
    tfim_results = run_tfim_bmera_sweep()
else:
    tfim_results = []
    print(
        'Legacy BMERA sweep disabled: analytic angles expose the control '
        'parameter too directly for a meaningful architecture comparison.'
    )


In [ ]:
# Visualize the leakage-control accuracy. Near-perfect linear performance is a
# warning sign here, not a positive BMERA result.
leakage_qubits = [row['n_qubits'] for row in tfim_leakage_control_rows]
plt.figure(figsize=(7, 4.5))
for metric, label in (
    ('validation', 'validation'),
    ('iid_test', 'IID test'),
    ('critical_test', 'near-critical test'),
):
    plt.plot(
        leakage_qubits,
        [row[metric] for row in tfim_leakage_control_rows],
        marker='o',
        label=label,
    )
plt.ylim(0.45, 1.02)
plt.xticks(leakage_qubits)
plt.xlabel('Bogoliubov modes')
plt.ylabel('balanced accuracy')
plt.title('Leakage control: linear ridge on analytic TFIM descriptors')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig('tfim_analytic_descriptor_leakage_control.png', dpi=160)
plt.show()


#### 7.2 Ground-State Measurement Snapshot Benchmarks

The primary physical benchmark uses exact ground states of three periodic spin
chains: TFIM, cluster-Ising, and XXZ. Each classifier receives only finite-shot
local X/Z measurement records encoded as RY angles. BMERA therefore processes
classical measurement data, not the entangled wavefunction itself.

Splits are made by Hamiltonian parameter and ground-state ID before measurements
are drawn. Validation and test snapshots can never be repeated views of a
training state. Predictions are averaged over snapshots belonging to the same
state, and model selection uses state-level validation balanced accuracy.

Long screening, confirmation, and 10,000-sample bootstrap runs are opt-in.


In [ ]:
from bmera.experiments.snapshot_suite import (
    SNAPSHOT_CACHE_DIR,
    SNAPSHOT_RESULT_DIR,
    build_snapshot_dataset_suite,
    print_snapshot_comparison,
    run_snapshot_benchmark_suite,
)

RUN_SNAPSHOT_SMOKE = False
RUN_SNAPSHOT_SCREEN = False
RUN_SNAPSHOT_CONFIRMATION = False

In [ ]:
# Fast generator/physics preview. The complete smoke suite is opt-in here and is
# also covered by automated tests in test_quantum_snapshot_benchmark.py.
snapshot_smoke_datasets = {}
snapshot_smoke_results = []
snapshot_smoke_ranking = []

if RUN_SNAPSHOT_SMOKE:
    snapshot_smoke_datasets = build_snapshot_dataset_suite(profile='smoke')
    for (model_name, n_qubits), dataset in snapshot_smoke_datasets.items():
        first_parameter = float(dataset['train']['parameter'][0])
        _, state, residual = qsb.ground_state(
            model_name,
            n_qubits,
            first_parameter,
            cache_dir=SNAPSHOT_CACHE_DIR,
        )
        observables = qsb.physical_observables(model_name, state, n_qubits)
        print(
            f'{model_name:>13s} | n={n_qubits} | residual={residual:.2e} | '
            f'observables={observables}'
        )
    snapshot_smoke_results, snapshot_smoke_ranking = (
        run_snapshot_benchmark_suite(snapshot_smoke_datasets, smoke=True)
    )
    print_snapshot_comparison(snapshot_smoke_results, snapshot_smoke_ranking)
else:
    print('Snapshot smoke cell skipped. Set RUN_SNAPSHOT_SMOKE=True to run it.')


In [ ]:
# Scientific sweeps are deliberately opt-in. Dataset generation is cached and
# resumable, so interrupted 16-qubit exact diagonalization does not lose work.
snapshot_screen_datasets = {}
snapshot_screen_results = []
snapshot_screen_ranking = []
snapshot_confirmation_datasets = {}
snapshot_confirmation_results = []
snapshot_confirmation_ranking = []

if RUN_SNAPSHOT_SCREEN:
    snapshot_screen_datasets = build_snapshot_dataset_suite(profile='screen')
    snapshot_screen_results, snapshot_screen_ranking = (
        run_snapshot_benchmark_suite(snapshot_screen_datasets)
    )
    print_snapshot_comparison(snapshot_screen_results, snapshot_screen_ranking)
    qsb.save_benchmark_results(
        snapshot_screen_results,
        snapshot_screen_ranking,
        SNAPSHOT_RESULT_DIR / 'screen',
    )
else:
    print('8-qubit snapshot screening skipped.')

if RUN_SNAPSHOT_CONFIRMATION:
    snapshot_confirmation_datasets = build_snapshot_dataset_suite(
        profile='confirmation'
    )
    snapshot_confirmation_results, snapshot_confirmation_ranking = (
        run_snapshot_benchmark_suite(snapshot_confirmation_datasets)
    )
    print_snapshot_comparison(
        snapshot_confirmation_results,
        snapshot_confirmation_ranking,
    )
    qsb.save_benchmark_results(
        snapshot_confirmation_results,
        snapshot_confirmation_ranking,
        SNAPSHOT_RESULT_DIR / 'confirmation',
    )
else:
    print('8/16-qubit snapshot confirmation skipped.')


In [ ]:
# Plot the most complete available benchmark. No winner is claimed when the
# bootstrap interval includes zero.
snapshot_plot_results = (
    snapshot_confirmation_results
    if snapshot_confirmation_results
    else snapshot_screen_results
)
snapshot_plot_ranking = (
    snapshot_confirmation_ranking
    if snapshot_confirmation_ranking
    else snapshot_screen_ranking
)

if not snapshot_plot_results:
    print('No snapshot benchmark results to plot.')
else:
    labels = []
    values = []
    errors = []
    for dataset_result in snapshot_plot_results:
        for model_name, result in dataset_result['models'].items():
            summary = qsb.summarize_model_result(result)
            labels.append(
                f"{dataset_result['dataset']}\nn={dataset_result['n_qubits']}\n"
                f"{model_name}"
            )
            values.append(
                summary['critical_test_state_balanced_accuracy_mean']
            )
            errors.append(
                summary['critical_test_state_balanced_accuracy_std']
            )
    x = np.arange(len(labels))
    plt.figure(figsize=(max(10, 0.65 * len(labels)), 5))
    plt.bar(x, values, yerr=errors, capsize=3)
    plt.axhline(0.5, color='black', linestyle='--', linewidth=1)
    plt.xticks(x, labels, rotation=45, ha='right')
    plt.ylim(0.0, 1.05)
    plt.ylabel('near-critical state balanced accuracy')
    plt.title('Ground-state measurement snapshot benchmark')
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig('quantum_snapshot_model_comparison.png', dpi=160)
    plt.show()

    plt.figure(figsize=(8, 4.5))
    rank_labels = [
        f"{row['dataset']} n={row['n_qubits']}"
        for row in snapshot_plot_ranking
    ]
    gaps = np.asarray([row['difference'] for row in snapshot_plot_ranking])
    lower = gaps - np.asarray([row['ci_lower'] for row in snapshot_plot_ranking])
    upper = np.asarray([row['ci_upper'] for row in snapshot_plot_ranking]) - gaps
    plt.errorbar(
        np.arange(len(gaps)),
        gaps,
        yerr=np.stack([lower, upper]),
        fmt='o',
        capsize=4,
    )
    plt.axhline(0.0, color='black', linestyle='--', linewidth=1)
    plt.xticks(np.arange(len(gaps)), rank_labels, rotation=25, ha='right')
    plt.ylabel('BMERA - best classical balanced accuracy')
    plt.title('Paired hierarchical bootstrap advantage')
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig('quantum_snapshot_bmera_advantage.png', dpi=160)
    plt.show()


### 8. Gradient Norm, Variance, and Signal-to-Noise Diagnostics

This section measures BMERA MSE-loss gradient statistics across qubit count, block type, layer, and red/green block color. The default sweep is intentionally small enough for interactive use; the optional paper sweep reuses the same helpers with larger trial and batch counts.


In [ ]:
from bmera.experiments.gradient_diagnostics import (
    GRAD_DIAG_BATCH_FAST,
    GRAD_DIAG_BATCH_PAPER,
    GRAD_DIAG_QUBITS,
    GRAD_DIAG_TRIALS_FAST,
    GRAD_DIAG_TRIALS_PAPER,
    GRAD_DIAG_TYPES,
    print_gradient_diag_global_table,
    print_gradient_diag_layer_table,
    run_gradient_diagnostics,
)

In [ ]:
RUN_GRADIENT_DIAGNOSTICS = False
grad_diag_param_rows_fast, grad_diag_group_rows_fast, grad_diag_global_rows_fast = [], [], []
if RUN_GRADIENT_DIAGNOSTICS:
    grad_diag_param_rows_fast, grad_diag_group_rows_fast, grad_diag_global_rows_fast, grad_diag_device_fast = run_gradient_diagnostics(
        qubits=GRAD_DIAG_QUBITS,
        block_types=GRAD_DIAG_TYPES,
        trials=GRAD_DIAG_TRIALS_FAST,
        batch_size=GRAD_DIAG_BATCH_FAST,
        seed=2026,
        prefer_gpu=True,
        label='fast',
    )

    print('\nGlobal gradient diagnostic summary')
    print_gradient_diag_global_table(grad_diag_global_rows_fast)
    print()
    print_gradient_diag_layer_table(
        grad_diag_group_rows_fast,
        n_qubits=max(GRAD_DIAG_QUBITS),
        block_type=GRAD_DIAG_TYPES[0],
    )


In [ ]:
if RUN_GRADIENT_DIAGNOSTICS:
    # Gradient diagnostic plots.

    def _grad_diag_series(rows, block_type, y_key):
        selected = sorted(
            [row for row in rows if row['block_type'] == int(block_type)],
            key=lambda row: row['n_qubits'],
        )
        return np.array([row['n_qubits'] for row in selected]), np.array([row[y_key] for row in selected])


    def plot_gradient_diag_global(rows, y_key, ylabel, title, filename=None, log_y=False):
        if not rows:
            print(f'Skipping plot {title}: no rows.')
            return
        plt.figure(figsize=(7, 4))
        for block_type in GRAD_DIAG_TYPES:
            x, y = _grad_diag_series(rows, block_type, y_key)
            if x.size == 0:
                continue
            plt.plot(x, y, marker='o', label=f'type {block_type}')
        plt.xscale('log', base=2)
        if log_y:
            plt.yscale('log')
        plt.xlabel('qubits')
        plt.ylabel(ylabel)
        plt.title(title)
        plt.grid(True, which='both', alpha=0.25)
        plt.legend()
        plt.tight_layout()
        if SAVE_FIGURES and filename:
            plt.savefig(filename, dpi=160)
        plt.show()


    plot_gradient_diag_global(
        grad_diag_global_rows_fast,
        'grad_norm_mean',
        'mean full-gradient L2 norm',
        'BMERA gradient norm scaling',
        filename='gradient_diagnostics_global_norm.png',
    )
    plot_gradient_diag_global(
        grad_diag_global_rows_fast,
        'mean_param_var',
        'mean parameter gradient variance',
        'BMERA gradient variance scaling',
        filename='gradient_diagnostics_variance.png',
        log_y=True,
    )
    plot_gradient_diag_global(
        grad_diag_global_rows_fast,
        'mean_param_snr',
        'mean parameter SNR',
        'BMERA gradient signal-to-noise scaling',
        filename='gradient_diagnostics_snr.png',
        log_y=True,
    )

    # Layer/color plot for the largest qubit count.
    layer_plot_qubits = max(GRAD_DIAG_QUBITS)
    fig, axes = plt.subplots(1, len(GRAD_DIAG_TYPES), figsize=(5 * len(GRAD_DIAG_TYPES), 4), sharey=True)
    if len(GRAD_DIAG_TYPES) == 1:
        axes = [axes]
    for ax, block_type in zip(axes, GRAD_DIAG_TYPES):
        for color in ('red', 'green'):
            selected = sorted(
                [
                    row for row in grad_diag_group_rows_fast
                    if row['n_qubits'] == int(layer_plot_qubits)
                    and row['block_type'] == int(block_type)
                    and row['color'] == color
                ],
                key=lambda row: row['layer'],
            )
            if not selected:
                continue
            ax.plot(
                [row['layer'] for row in selected],
                [row['mean_param_snr'] for row in selected],
                marker='o',
                label=color,
            )
        ax.set_title(f'type {block_type}, n={layer_plot_qubits}')
        ax.set_xlabel('layer')
        ax.set_yscale('log')
        ax.grid(True, which='both', alpha=0.25)
        ax.legend()
    axes[0].set_ylabel('mean parameter SNR')
    fig.suptitle('BMERA layer/color gradient SNR')
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig('gradient_diagnostics_layer_color.png', dpi=160)
    plt.show()


In [ ]:
# Optional paper-grade diagnostic sweep.
# This can be much slower because it compiles and evaluates all requested qubit/type pairs.
RUN_GRAD_DIAG_PAPER_SWEEP = False

if RUN_GRAD_DIAG_PAPER_SWEEP:
    grad_diag_param_rows_paper, grad_diag_group_rows_paper, grad_diag_global_rows_paper, grad_diag_device_paper = run_gradient_diagnostics(
        qubits=GRAD_DIAG_QUBITS,
        block_types=GRAD_DIAG_TYPES,
        trials=GRAD_DIAG_TRIALS_PAPER,
        batch_size=GRAD_DIAG_BATCH_PAPER,
        seed=3026,
        prefer_gpu=True,
        label='paper',
    )
    print_gradient_diag_global_table(grad_diag_global_rows_paper)
else:
    grad_diag_param_rows_paper = []
    grad_diag_group_rows_paper = []
    grad_diag_global_rows_paper = []
    grad_diag_device_paper = None
    print('Paper-grade gradient diagnostics skipped. Set RUN_GRAD_DIAG_PAPER_SWEEP=True to run the larger sweep.')


#### 8.1 Moons Early-Training Gradient Diagnostics

This subsection remeasures gradient variance, SNR, norm, and mean absolute gradient along an actual Adam training path on the `moons` dataset. Checkpoints are dense in the first 100 steps because that is where the moons model changes most rapidly.


In [ ]:
from bmera.experiments.gradient_diagnostics import (
    MOONS_PATH_EPS,
    print_moons_path_global_table,
    run_moons_training_path_diagnostics,
)

RUN_MOONS_PATH_SMOKE = False

In [ ]:
RUN_MOONS_PATH = False
moons_path_global_rows, moons_path_heatmap_data = [], {}
if RUN_MOONS_PATH or RUN_MOONS_PATH_SMOKE:
    moons_path_param_rows, moons_path_group_rows, moons_path_global_rows, moons_path_heatmap_data, moons_path_device = run_moons_training_path_diagnostics(
        smoke=RUN_MOONS_PATH_SMOKE
    )

    print('\nMoons early-training gradient diagnostic summary')
    print_moons_path_global_table(moons_path_global_rows)


In [ ]:
if moons_path_global_rows:
    # Moons early-training gradient diagnostic plots.

    def _moons_path_series(rows, n_qubits, y_key):
        selected = sorted([row for row in rows if row['n_qubits'] == int(n_qubits)], key=lambda row: row['step'])
        return np.array([row['step'] for row in selected]), np.array([row[y_key] for row in selected])


    def plot_moons_path_metric(rows, y_key, ylabel, title, filename=None, log_y=True):
        if not rows:
            print(f'Skipping {title}: no rows.')
            return
        plt.figure(figsize=(7, 4))
        for n_qubits in sorted({row['n_qubits'] for row in rows}):
            x, y = _moons_path_series(rows, n_qubits, y_key)
            plt.plot(x, y, marker='o', label=f'{n_qubits} qubits')
        if log_y:
            plt.yscale('log')
        plt.xlabel('Adam training step')
        plt.ylabel(ylabel)
        plt.title(title)
        plt.grid(True, which='both', alpha=0.25)
        plt.legend()
        plt.tight_layout()
        if SAVE_FIGURES and filename:
            plt.savefig(filename, dpi=160)
        plt.show()


    plot_moons_path_metric(
        moons_path_global_rows,
        'mean_param_var',
        'mean parameter gradient variance',
        'Moons early-training gradient variance',
        filename='moons_early_training_gradient_variance.png',
    )
    plot_moons_path_metric(
        moons_path_global_rows,
        'mean_param_snr',
        'mean parameter SNR',
        'Moons early-training gradient SNR',
        filename='moons_early_training_gradient_snr.png',
    )
    plot_moons_path_metric(
        moons_path_global_rows,
        'grad_norm_mean',
        'mean full-gradient L2 norm',
        'Moons early-training gradient norm',
        filename='moons_early_training_gradient_norm.png',
    )

    final_step = max(row['step'] for row in moons_path_global_rows)
    heatmap_qubits = sorted({key[0] for key in moons_path_heatmap_data if key[1] == final_step})
    fig, axes = plt.subplots(1, len(heatmap_qubits), figsize=(7 * len(heatmap_qubits), 4), squeeze=False)
    for ax, n_qubits in zip(axes[0], heatmap_qubits):
        heatmap = moons_path_heatmap_data[(int(n_qubits), int(final_step))]
        positive = heatmap[heatmap > 0]
        floor = float(np.min(positive)) if positive.size else MOONS_PATH_EPS
        image = ax.imshow(np.log10(np.maximum(heatmap, floor)), aspect='auto', origin='lower', cmap='magma')
        ax.set_title(f'n={n_qubits}, step={final_step}')
        ax.set_xlabel('flattened per-layer parameter index')
        ax.set_ylabel('layer')
        fig.colorbar(image, ax=ax, label='log10(mean abs grad)')
    fig.suptitle('Moons early-training mean absolute gradient heatmap')
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig('moons_early_training_mean_abs_heatmap_final.png', dpi=160)
    plt.show()


In [ ]:
# Optional: draw heatmaps for every recorded snapshot step.
RUN_MOONS_PATH_ALL_HEATMAPS = False

if RUN_MOONS_PATH_ALL_HEATMAPS:
    for n_qubits in sorted({key[0] for key in moons_path_heatmap_data}):
        steps = sorted(step for q, step in moons_path_heatmap_data if q == int(n_qubits))
        ncols = min(4, len(steps))
        nrows = int(np.ceil(len(steps) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.2 * nrows), squeeze=False)
        for ax in axes.ravel():
            ax.axis('off')
        for ax, step in zip(axes.ravel(), steps):
            heatmap = moons_path_heatmap_data[(int(n_qubits), int(step))]
            positive = heatmap[heatmap > 0]
            floor = float(np.min(positive)) if positive.size else MOONS_PATH_EPS
            image = ax.imshow(np.log10(np.maximum(heatmap, floor)), aspect='auto', origin='lower', cmap='magma')
            ax.set_title(f'step={step}')
            ax.set_xlabel('param index')
            ax.set_ylabel('layer')
            ax.axis('on')
        fig.suptitle(f'Moons mean abs gradient heatmaps, n={n_qubits}')
        fig.tight_layout()
        if SAVE_FIGURES:
            fig.savefig(f'moons_early_training_mean_abs_heatmaps_n{n_qubits}.png', dpi=160)
        plt.show()
else:
    print('All-snapshot moons heatmaps skipped. Set RUN_MOONS_PATH_ALL_HEATMAPS=True to render them.')


#### 8.2 Spin-Model Snapshot Early-Training Gradient Diagnostics

This section repeats the early-training diagnostics for TFIM, cluster-Ising, and
XXZ measurement snapshots at 8 and 16 qubits. Variance and SNR here are measured
over fixed measurement mini-batches at a fixed model state; they quantify
finite-data/batch gradient noise, not initialization-induced barren plateaus.

Reported summaries include vector SNR, median parameter SNR, normalized gradient
RMS, parameter variance, and mean-absolute-gradient heatmaps. Long runs are
opt-in and use the same cached, state-disjoint datasets as Section 7.2.


In [ ]:
from bmera.experiments.snapshot_suite import (
    SNAPSHOT_PATH_MODELS,
    SNAPSHOT_PATH_QUBITS,
    print_snapshot_path_table,
    run_all_snapshot_path_diagnostics,
)

RUN_SNAPSHOT_PATH_SMOKE = False
RUN_SNAPSHOT_PATH_DIAGNOSTICS = False

In [ ]:
snapshot_path_param_rows = []
snapshot_path_group_rows = []
snapshot_path_global_rows = []
snapshot_path_heatmap_data = {}
snapshot_path_devices = {}

if RUN_SNAPSHOT_PATH_SMOKE or RUN_SNAPSHOT_PATH_DIAGNOSTICS:
    (
        snapshot_path_param_rows,
        snapshot_path_group_rows,
        snapshot_path_global_rows,
        snapshot_path_heatmap_data,
        snapshot_path_devices,
    ) = run_all_snapshot_path_diagnostics(
        smoke=RUN_SNAPSHOT_PATH_SMOKE and not RUN_SNAPSHOT_PATH_DIAGNOSTICS
    )
    print_snapshot_path_table(snapshot_path_global_rows)
else:
    print(
        'Snapshot path diagnostics skipped. Set RUN_SNAPSHOT_PATH_SMOKE=True '
        'for a one-step test or RUN_SNAPSHOT_PATH_DIAGNOSTICS=True for all models.'
    )


In [ ]:
# Early-training curves for every generated model/qubit pair.
if not snapshot_path_global_rows:
    print('No snapshot path rows to plot.')
else:
    metric_specs = (
        ('mean_parameter_variance', 'mean parameter variance', True),
        ('median_parameter_snr', 'median parameter SNR', True),
        ('vector_snr', 'vector SNR', True),
        ('normalized_gradient_rms', 'normalized gradient RMS', True),
    )
    for metric, ylabel, log_scale in metric_specs:
        plt.figure(figsize=(8, 4.8))
        for model_name in SNAPSHOT_PATH_MODELS:
            for n_qubits in SNAPSHOT_PATH_QUBITS:
                selected = sorted(
                    [
                        row for row in snapshot_path_global_rows
                        if row['dataset'] == model_name
                        and row['n_qubits'] == n_qubits
                    ],
                    key=lambda row: row['step'],
                )
                if selected:
                    plt.plot(
                        [row['step'] for row in selected],
                        [row[metric] for row in selected],
                        marker='o',
                        label=f'{model_name}, n={n_qubits}',
                    )
        if log_scale:
            plt.yscale('log')
        plt.xlabel('training step')
        plt.ylabel(ylabel)
        plt.title(f'Snapshot batch-noise diagnostics: {ylabel}')
        plt.grid(alpha=0.25)
        plt.legend(ncol=2, fontsize=8)
        plt.tight_layout()
        if SAVE_FIGURES:
            plt.savefig(f'quantum_snapshot_path_{metric}.png', dpi=160)
        plt.show()


In [ ]:
# Mean-absolute-gradient heatmaps at the final available checkpoint.
if not snapshot_path_heatmap_data:
    print('No snapshot path heatmaps to render.')
else:
    keys = sorted(snapshot_path_heatmap_data)
    pairs = sorted({(model, n_qubits) for model, n_qubits, _ in keys})
    fig, axes = plt.subplots(
        len(pairs),
        1,
        figsize=(12, max(3, 2.8 * len(pairs))),
        squeeze=False,
    )
    for axis, (model_name, n_qubits) in zip(axes[:, 0], pairs):
        steps = [
            step for model, n, step in keys
            if model == model_name and n == n_qubits
        ]
        final_step = max(steps)
        heatmap = snapshot_path_heatmap_data[
            (model_name, n_qubits, final_step)
        ]
        image = axis.imshow(
            np.log10(np.maximum(heatmap, 1e-9)),
            aspect='auto',
            interpolation='nearest',
            cmap='magma',
        )
        axis.set_title(
            f'{model_name}, n={n_qubits}, step={final_step}: '
            'log10 mean |gradient|'
        )
        axis.set_ylabel('BMERA layer')
        axis.set_xlabel('flattened within-layer parameter')
        fig.colorbar(image, ax=axis, fraction=0.02, pad=0.02)
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(
            'quantum_snapshot_path_mean_abs_gradient_heatmaps.png',
            dpi=160,
        )
    plt.show()


### 9. GPU Batch-Size Sweeps: 8, 16, 32, ... and 256-Qubit Autodiff Gradients

This section measures steady-state autodiff gradient time for 8, 16, 32, and 64 qubits across batch sizes on the JAX GPU device. For each qubit count, the selected batch size is based on gradient throughput, with the maximum near-optimal batch defined as the largest batch whose throughput is at least `OPTIMAL_THROUGHPUT_FRACTION` of the measured peak.


In [ ]:
RUN_GPU_SWEEPS = False
from bmera.experiments.gpu_benchmark import (
    GPU_BATCH_SWEEP,
    GPU_SWEEP_QUBITS,
    OPTIMAL_THROUGHPUT_FRACTION,
    benchmark_autodiff_grad_batchsize_gpu,
    get_jax_gpu_devices,
)

gpu_batch_rows_by_qubits = {}
gpu_batch_failures_by_qubits = {}

for n_qubits in GPU_SWEEP_QUBITS if RUN_GPU_SWEEPS else ():
    rows, failures = benchmark_autodiff_grad_batchsize_gpu(n_qubits)
    gpu_batch_rows_by_qubits[int(n_qubits)] = rows
    gpu_batch_failures_by_qubits[int(n_qubits)] = failures

gpu_batch_rows_by_qubits

In [ ]:
from bmera.experiments.gpu_benchmark import (
    summarize_gpu_batch_sweep,
)

gpu_batch_summaries_by_qubits = {}
for n_qubits in GPU_SWEEP_QUBITS:
    print('-' * 80)
    gpu_batch_summaries_by_qubits[int(n_qubits)] = summarize_gpu_batch_sweep(
        gpu_batch_rows_by_qubits.get(int(n_qubits), []),
        gpu_batch_failures_by_qubits.get(int(n_qubits), []),
    )

gpu_batch_summaries_by_qubits

In [ ]:
if RUN_GPU_SWEEPS:
    # Optional sweep: determine the best fixed microbatch size for large logical batches.
    #
    # Selection rule:
    # - `best_peak_microbatch_by_qubits` gives the absolute fastest candidate.
    # - `recommended_microbatch_by_qubits` keeps peak throughput within
    #   OPTIMAL_THROUGHPUT_FRACTION of the best candidate, then chooses the one
    #   with the largest successful logical batch. This is usually the better
    #   training default when memory stability matters.

    GPU_MICROBATCH_QUBITS = GPU_SWEEP_QUBITS
    GPU_MICROBATCH_CANDIDATES = (None, 32, 64, 128, 256)
    GPU_MICROBATCH_BATCH_SWEEP = GPU_BATCH_SWEEP
    GPU_MICROBATCH_REPEATS = 3
    GPU_MICROBATCH_WARMUP = 1

    def microbatch_label(microbatch_size):
        return 'full' if microbatch_size is None else str(int(microbatch_size))

    def print_microbatch_summary_table(n_qubits, summaries):
        print(f'\n{n_qubits}-qubit microbatch sweep summary')
        print('micro | peak batch | peak samples/s | largest ok batch | first failed | max effective micro')
        print('-' * 88)
        for summary in summaries:
            if summary.get('status') != 'ok':
                print(f"{summary['requested_microbatch_label']:>5} | no data: {summary.get('message', 'benchmark failed')}")
                continue
            print(
                f"{summary['requested_microbatch_label']:>5} | "
                f"{summary['peak_batch_size']:10d} | "
                f"{summary['peak_grad_samples_per_second']:14.1f} | "
                f"{summary['largest_successful_batch_size']:16d} | "
                f"{str(summary['first_failed_batch_size']):>12} | "
                f"{summary['largest_effective_microbatch_size']:19d}"
            )

    gpu_microbatch_rows_by_qubits = {}
    gpu_microbatch_failures_by_qubits = {}
    gpu_microbatch_summaries_by_qubits = {}
    best_peak_microbatch_by_qubits = {}
    recommended_microbatch_by_qubits = {}

    for n_qubits in GPU_MICROBATCH_QUBITS:
        candidate_summaries = []
        gpu_microbatch_rows_by_qubits[int(n_qubits)] = {}
        gpu_microbatch_failures_by_qubits[int(n_qubits)] = {}

        for microbatch_size in GPU_MICROBATCH_CANDIDATES:
            label = microbatch_label(microbatch_size)
            print('=' * 88)
            print(f'Benchmarking n={n_qubits}, requested microbatch={label}')
            rows, failures = benchmark_autodiff_grad_batchsize_gpu(
                n_qubits,
                batch_sizes=GPU_MICROBATCH_BATCH_SWEEP,
                repeats=GPU_MICROBATCH_REPEATS,
                warmup=GPU_MICROBATCH_WARMUP,
                seed=1000 + int(n_qubits),
                microbatch_size=microbatch_size,
            )

            gpu_microbatch_rows_by_qubits[int(n_qubits)][label] = rows
            gpu_microbatch_failures_by_qubits[int(n_qubits)][label] = failures

            summary = summarize_gpu_batch_sweep(rows, failures)
            summary['requested_microbatch_size'] = microbatch_size
            summary['requested_microbatch_label'] = label
            summary['largest_effective_microbatch_size'] = max(
                (int(r['effective_microbatch_size']) for r in rows),
                default=0,
            )
            candidate_summaries.append(summary)

        ok_summaries = [s for s in candidate_summaries if s.get('status') == 'ok']
        gpu_microbatch_summaries_by_qubits[int(n_qubits)] = candidate_summaries
        print_microbatch_summary_table(n_qubits, candidate_summaries)

        if ok_summaries:
            best_peak = max(
                ok_summaries,
                key=lambda s: (s['peak_grad_samples_per_second'], s['largest_successful_batch_size']),
            )
            throughput_cutoff = OPTIMAL_THROUGHPUT_FRACTION * best_peak['peak_grad_samples_per_second']
            near_best = [s for s in ok_summaries if s['peak_grad_samples_per_second'] >= throughput_cutoff]
            recommended = max(
                near_best,
                key=lambda s: (s['largest_successful_batch_size'], s['peak_grad_samples_per_second']),
            )

            best_peak_microbatch_by_qubits[int(n_qubits)] = best_peak
            recommended_microbatch_by_qubits[int(n_qubits)] = recommended
            print(
                f"\nBest peak throughput for n={n_qubits}: micro={best_peak['requested_microbatch_label']}, "
                f"batch={best_peak['peak_batch_size']}, "
                f"{best_peak['peak_grad_samples_per_second']:.1f} samples/s"
            )
            print(
                f"Recommended training microbatch for n={n_qubits}: micro={recommended['requested_microbatch_label']}, "
                f"largest successful logical batch={recommended['largest_successful_batch_size']}, "
                f"peak={recommended['peak_grad_samples_per_second']:.1f} samples/s"
            )

    recommended_microbatch_by_qubits


In [ ]:
for n_qubits in GPU_SWEEP_QUBITS:
    rows = gpu_batch_rows_by_qubits.get(int(n_qubits), [])
    summary = gpu_batch_summaries_by_qubits.get(int(n_qubits), {})
    if not rows:
        print(f'Skipping plot for {n_qubits} qubits: {summary.get("message", "no GPU benchmark data")}.')
        continue

    batch_sizes = np.array([r['batch_size'] for r in rows])
    grad_ms = np.array([r['grad_steady_ms'] for r in rows])
    grad_std_ms = np.array([r['grad_steady_std_ms'] for r in rows])
    throughput = np.array([r['grad_samples_per_second'] for r in rows])

    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.errorbar(batch_sizes, grad_ms, yerr=grad_std_ms, marker='o', label='gradient time')
    ax1.set_xscale('log', base=2)
    ax1.set_xlabel('batch size')
    ax1.set_ylabel('autodiff gradient time, ms')

    ax2 = ax1.twinx()
    ax2.plot(batch_sizes, throughput, marker='s', color='tab:green', label='throughput')
    ax2.set_ylabel('gradient throughput, samples/s')

    peak_batch = summary.get('peak_batch_size')
    max_opt_batch = summary.get('max_near_optimal_batch_size')
    if peak_batch is not None:
        ax1.axvline(peak_batch, color='tab:green', linestyle=':', alpha=0.75, label='peak throughput')
    if max_opt_batch is not None:
        ax1.axvline(max_opt_batch, color='tab:red', linestyle='--', alpha=0.75, label='max near-optimal batch')

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best')
    plt.title(f'{n_qubits}-qubit autodiff gradient scaling on GPU')
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(f"{n_qubits}_qubits_gradient_throughput_upd.png")


### 10. TensorCircuit Backend Benchmark: 8-32 Qubits

This section validates the TensorCircuit backend against the native BMERA simulator for block types 1, 2, and 3. Timing is kept on type 1 by default because TensorCircuit compilation can be slow; set `TENSORCIRCUIT_TIMING_TYPES = (1, 2, 3)` after the type-1 run looks reasonable.


In [ ]:
import json

from bmera_core import (
    tensorcircuit_forward_batch,
    tensorcircuit_loss,
    loss_and_grad_tensorcircuit,
    validate_tensorcircuit_backend,
    benchmark_tensorcircuit_backend_scaling,
)

TENSORCIRCUIT_BENCH_TYPES = (1, 2, 3)
TENSORCIRCUIT_TIMING_TYPES = (1,)  # Change to (1, 2, 3) when longer timing runs are acceptable.
TENSORCIRCUIT_VALIDATION_QUBITS = (4, 8)
TENSORCIRCUIT_TIMING_QUBITS = (8, 16, 32)
TENSORCIRCUIT_TIMING_BATCHES = (1, 4, 16)
TENSORCIRCUIT_REPEATS = 3
TENSORCIRCUIT_WARMUP = 1

RUN_TENSORCIRCUIT_VALIDATION = False
RUN_TENSORCIRCUIT_SMOKE = False
RUN_TENSORCIRCUIT_TIMING = False


In [ ]:
# Forward/loss/gradient agreement against the native simulator for all BMERA block types.
if RUN_TENSORCIRCUIT_VALIDATION:
    tensorcircuit_validation_rows = validate_tensorcircuit_backend(
        n_qubits_list=TENSORCIRCUIT_VALIDATION_QUBITS,
        types=TENSORCIRCUIT_BENCH_TYPES,
        batch_size=2,
        seed=20260502,
    )

    print('type | n_qubits | forward_max_abs_error | loss_abs_error | grad_max_abs_error')
    for row in tensorcircuit_validation_rows:
        print(
            f"{row['type']:>4d} | {row['n_qubits']:>8d} | "
            f"{row['forward_max_abs_error']:.3e} | "
            f"{row['loss_abs_error']:.3e} | "
            f"{row['grad_max_abs_error']:.3e}"
        )
else:
    tensorcircuit_validation_rows = []


In [ ]:
# Smoke-test compilation and one measured pass for every block type before running larger timings.
if RUN_TENSORCIRCUIT_SMOKE:
    tensorcircuit_smoke_rows = benchmark_tensorcircuit_backend_scaling(
        n_qubits_list=(8,),
        batch_sizes=(1,),
        types=TENSORCIRCUIT_BENCH_TYPES,
        repeats=1,
        warmup=0,
        seed=20260503,
    )
    for row in tensorcircuit_smoke_rows:
        print(
            f"type={row['type']} n={row['n_qubits']} batch={row['batch_size']} | "
            f"native grad {row['existing_grad_steady_ms']:.3f} ms | "
            f"TensorCircuit grad {row['tensorcircuit_grad_steady_ms']:.3f} ms"
        )
else:
    tensorcircuit_smoke_rows = []


In [ ]:
# Default timing comparison: type 1 over 8-32 qubits.
# TensorCircuit compilation is intentionally reported separately from steady-state timing.
if RUN_TENSORCIRCUIT_TIMING:
    tensorcircuit_benchmark_rows = benchmark_tensorcircuit_backend_scaling(
        n_qubits_list=TENSORCIRCUIT_TIMING_QUBITS,
        batch_sizes=TENSORCIRCUIT_TIMING_BATCHES,
        types=TENSORCIRCUIT_TIMING_TYPES,
        repeats=TENSORCIRCUIT_REPEATS,
        warmup=TENSORCIRCUIT_WARMUP,
        seed=20260504,
    )
    with open('tensorcircuit_backend_benchmark_rows.json', 'w') as f:
        json.dump(tensorcircuit_benchmark_rows, f, indent=2)
else:
    tensorcircuit_benchmark_rows = []


In [ ]:
def plot_tensorcircuit_backend_comparison(rows, metric, ylabel, title, filename):
    if not rows:
        print(f'Skipping {title}: no TensorCircuit benchmark rows.')
        return

    selected_type = TENSORCIRCUIT_TIMING_TYPES[0]
    selected_batch = min(row['batch_size'] for row in rows if row['type'] == selected_type)
    selected = sorted(
        [row for row in rows if row['type'] == selected_type and row['batch_size'] == selected_batch],
        key=lambda row: row['n_qubits'],
    )
    if not selected:
        print(f'Skipping {title}: no rows for type={selected_type}, batch={selected_batch}.')
        return

    n_values = np.array([row['n_qubits'] for row in selected])
    native = np.array([row[f'existing_{metric}_steady_ms'] for row in selected])
    tc = np.array([row[f'tensorcircuit_{metric}_steady_ms'] for row in selected])

    plt.figure(figsize=(7, 4.5))
    plt.plot(n_values, native, 'o-', label='native BMERA simulator')
    plt.plot(n_values, tc, 's-', label='TensorCircuit backend')
    plt.xscale('log', base=2)
    plt.yscale('log')
    plt.xlabel('number of qubits')
    plt.ylabel(ylabel)
    plt.title(f'{title} (type {selected_type}, batch {selected_batch})')
    plt.grid(True, which='both', alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.show()

plot_tensorcircuit_backend_comparison(
    tensorcircuit_benchmark_rows,
    metric='forward',
    ylabel='steady forward time, ms',
    title='Forward-pass backend comparison',
    filename='tensorcircuit_vs_bmera_forward_time.png',
)
plot_tensorcircuit_backend_comparison(
    tensorcircuit_benchmark_rows,
    metric='grad',
    ylabel='steady loss+gradient time, ms',
    title='Backward-pass backend comparison',
    filename='tensorcircuit_vs_bmera_grad_time.png',
)
